Import libraries and set project paths

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

# Project directory
PROJECT_DIR = Path.cwd().parent

# Data directories
RAW_DIR = PROJECT_DIR / "data" / "raw"
PROCESSED_DIR = PROJECT_DIR / "data" / "processed"

# Create processed folder if it does not exist
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

# Dataset path
DATA_PATH = RAW_DIR / "employee_promotion.csv"

print("Project directory:", PROJECT_DIR)
print("Raw directory:", RAW_DIR)
print("Dataset path:", DATA_PATH)

assert DATA_PATH.is_file(), "Dataset not found. Check the file location."

print("\nDataset file found successfully.")

Project directory: c:\Users\M S I\VSCodeProject\AI Engineering Project\ai-engineering-talentrank-main
Raw directory: c:\Users\M S I\VSCodeProject\AI Engineering Project\ai-engineering-talentrank-main\data\raw
Dataset path: c:\Users\M S I\VSCodeProject\AI Engineering Project\ai-engineering-talentrank-main\data\raw\employee_promotion.csv

Dataset file found successfully.


Load the CSV file

In [2]:
# Load the original dataset
df = pd.read_csv(DATA_PATH, low_memory=False)

print("Dataset loaded successfully!")
print("Dataset shape:", df.shape)

Dataset loaded successfully!
Dataset shape: (54808, 13)


Inspect the first five records

In [3]:
df.head()

,employee_id,department,region,education,gender,recruitment_channel,no_of_trainings,age,previous_year_rating,length_of_service,awards_won,avg_training_score,is_promoted
0,65438,Sales & Marketing,region_7,Master's & above,f,sourcing,1,35,5.0,8,0,49.0,0
1,65141,Operations,region_22,Bachelor's,m,other,1,30,5.0,4,0,60.0,0
2,7513,Sales & Marketing,region_19,Bachelor's,m,sourcing,1,34,3.0,7,0,50.0,0
3,2542,Sales & Marketing,region_23,Bachelor's,m,other,2,39,1.0,10,0,50.0,0
4,48945,Technology,region_26,Bachelor's,m,other,1,45,3.0,2,0,73.0,0


Inspect the last five records

In [4]:
df.tail()

,employee_id,department,region,education,gender,recruitment_channel,no_of_trainings,age,previous_year_rating,length_of_service,awards_won,avg_training_score,is_promoted
54803,3030,Technology,region_14,Bachelor's,m,sourcing,1,48,3.0,17,0,78.0,0
54804,74592,Operations,region_27,Master's & above,f,other,1,37,2.0,6,0,56.0,0
54805,13918,Analytics,region_1,Bachelor's,m,other,1,27,5.0,3,0,79.0,0
54806,13614,Sales & Marketing,region_9,NaN,m,sourcing,1,29,1.0,2,0,NaN,0
54807,51526,HR,region_22,Bachelor's,m,other,1,27,1.0,5,0,49.0,0


Inspect column names and data types

In [5]:
print("Column names:")
print(df.columns.tolist())

print("\nDataset information:")
df.info()

Column names:
['employee_id', 'department', 'region', 'education', 'gender', 'recruitment_channel', 'no_of_trainings', 'age', 'previous_year_rating', 'length_of_service', 'awards_won', 'avg_training_score', 'is_promoted']

Dataset information:
<class 'pandas.DataFrame'>
RangeIndex: 54808 entries, 0 to 54807
Data columns (total 13 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   employee_id           54808 non-null  int64  
 1   department            54808 non-null  str    
 2   region                54808 non-null  str    
 3   education             52399 non-null  str    
 4   gender                54808 non-null  str    
 5   recruitment_channel   54808 non-null  str    
 6   no_of_trainings       54808 non-null  int64  
 7   age                   54808 non-null  int64  
 8   previous_year_rating  50684 non-null  float64
 9   length_of_service     54808 non-null  int64  
 10  awards_won            54808 non-null  int

Check missing values

In [6]:
missing_values = df.isnull().sum()

missing_summary = pd.DataFrame({
    "Missing Values": missing_values,
    "Missing Percentage": (missing_values / len(df)) * 100
})

missing_summary

,Missing Values,Missing Percentage
employee_id,0,0.000000
department,0,0.000000
region,0,0.000000
education,2409,4.395344
gender,0,0.000000
recruitment_channel,0,0.000000
no_of_trainings,0,0.000000
age,0,0.000000
previous_year_rating,4124,7.524449
length_of_service,0,0.000000


Check duplicate records

In [7]:
print("Total duplicate records:", df.duplicated().sum())

Total duplicate records: 0


Inspect the promotion outcome

In [8]:
if "is_promoted" in df.columns:
    print("Promotion outcome distribution:")
    print(df["is_promoted"].value_counts(dropna=False))

    print("\nPromotion outcome percentage:")
    print(
        df["is_promoted"]
        .value_counts(normalize=True, dropna=False) * 100
    )
else:
    print("Column 'is_promoted' was not found.")
    print("Please check the actual column names.")

Promotion outcome distribution:
is_promoted
0    50140
1     4668
Name: count, dtype: int64

Promotion outcome percentage:
is_promoted
0    91.482995
1     8.517005
Name: proportion, dtype: float64


Check employee ID uniqueness

In [9]:
# Check whether employee_id is unique
total_employee_ids = df["employee_id"].count()
unique_employee_ids = df["employee_id"].nunique()

print("Total employee IDs:", total_employee_ids)
print("Unique employee IDs:", unique_employee_ids)
print("Duplicate employee IDs:", total_employee_ids - unique_employee_ids)

Total employee IDs: 54808
Unique employee IDs: 54808
Duplicate employee IDs: 0


Investigate missing previous-year ratings

In [10]:
# Compare missing ratings with length of service
rating_missing = df[df["previous_year_rating"].isnull()]

print("Records with missing previous-year rating:")
print("Total:", len(rating_missing))

print("\nLength of service distribution:")
print(rating_missing["length_of_service"].value_counts().sort_index())

Records with missing previous-year rating:
Total: 4124

Length of service distribution:
length_of_service
1    4124
Name: count, dtype: int64


Check text whitespace

In [11]:
# Check leading and trailing whitespace in text columns

text_columns = df.select_dtypes(
    include=["object", "string"]
).columns

whitespace_summary = []

for column in text_columns:
    values = df[column].dropna().astype("string")

    whitespace_count = (
        values != values.str.strip()
    ).sum()

    whitespace_summary.append({
        "Column": column,
        "Values Checked": len(values),
        "Values With Leading/Trailing Whitespace": int(whitespace_count)
    })

whitespace_df = pd.DataFrame(whitespace_summary)

whitespace_df

,Column,Values Checked,Values With Leading/Trailing Whitespace
0,department,54808,0
1,region,54808,0
2,education,52399,0
3,gender,54808,0
4,recruitment_channel,54808,0


Compare original and prepared configurations

In [12]:
# Approach A: Original dataset
approach_a = df.copy()

# Approach B: Remove exact duplicates and trim text
approach_b = df.drop_duplicates().copy()

for column in approach_b.select_dtypes(
    include=["object", "string"]
).columns:
    approach_b[column] = (
        approach_b[column]
        .astype("string")
        .str.strip()
    )

# Compare both approaches
comparison = pd.DataFrame([
    {
        "Measurement": "Number of records",
        "Approach A - Original": len(approach_a),
        "Approach B - Prepared": len(approach_b)
    },
    {
        "Measurement": "Number of columns",
        "Approach A - Original": len(approach_a.columns),
        "Approach B - Prepared": len(approach_b.columns)
    },
    {
        "Measurement": "Duplicate rows",
        "Approach A - Original": int(approach_a.duplicated().sum()),
        "Approach B - Prepared": int(approach_b.duplicated().sum())
    },
    {
        "Measurement": "Missing values",
        "Approach A - Original": int(approach_a.isna().sum().sum()),
        "Approach B - Prepared": int(approach_b.isna().sum().sum())
    }
])

comparison

,Measurement,Approach A - Original,Approach B - Prepared
0,Number of records,54808,54808
1,Number of columns,13,13
2,Duplicate rows,0,0
3,Missing values,9093,9093


Export the prepared dataset

In [13]:
# Export the prepared dataset

PREPARED_DATA_PATH = PROCESSED_DIR / "prepared_table.csv"

approach_b.to_csv(PREPARED_DATA_PATH, index=False)

print("Prepared dataset exported successfully.")
print("Output path:", PREPARED_DATA_PATH)
print("Dataset shape:", approach_b.shape)

Prepared dataset exported successfully.
Output path: c:\Users\M S I\VSCodeProject\AI Engineering Project\ai-engineering-talentrank-main\data\processed\prepared_table.csv
Dataset shape: (54808, 13)


Validate the exported file

In [14]:
# Validate the exported dataset

prepared_df = pd.read_csv(PREPARED_DATA_PATH)

print("File exists:", PREPARED_DATA_PATH.exists())
print("Original shape:", df.shape)
print("Prepared shape:", prepared_df.shape)
print("Missing values:", prepared_df.isna().sum().sum())
print("Duplicate rows:", prepared_df.duplicated().sum())

assert prepared_df.shape == df.shape
assert prepared_df.isna().sum().sum() == 9093
assert prepared_df.duplicated().sum() == 0

print("\nValidation passed.")

File exists: True
Original shape: (54808, 13)
Prepared shape: (54808, 13)
Missing values: 9093
Duplicate rows: 0

Validation passed.


Record data lineage

In [15]:
# Record data lineage and file fingerprints

import hashlib
import json

MANIFEST_PATH = PROJECT_DIR / "data" / "week3_manifest.json"


def calculate_sha256(file_path):
    sha256 = hashlib.sha256()

    with open(file_path, "rb") as file:
        for chunk in iter(lambda: file.read(4096), b""):
            sha256.update(chunk)

    return sha256.hexdigest()


manifest = {
    "project": "TalentRank",
    "pipeline_version": "V1",
    "source": {
        "file": str(DATA_PATH.relative_to(PROJECT_DIR)),
        "sha256": calculate_sha256(DATA_PATH),
        "records": int(df.shape[0]),
        "columns": int(df.shape[1])
    },
    "output": {
        "file": str(PREPARED_DATA_PATH.relative_to(PROJECT_DIR)),
        "sha256": calculate_sha256(PREPARED_DATA_PATH),
        "records": int(prepared_df.shape[0]),
        "columns": int(prepared_df.shape[1]),
        "missing_values": int(prepared_df.isna().sum().sum()),
        "duplicate_rows": int(prepared_df.duplicated().sum())
    },
    "transformations": [
        "Loaded the original CSV dataset.",
        "Checked for exact duplicate rows.",
        "Checked text columns for leading and trailing whitespace.",
        "Removed exact duplicate rows if present.",
        "Trimmed leading and trailing whitespace from text columns.",
        "Preserved missing values for later model-specific preprocessing."
    ],
    "notes": [
        "No duplicate rows were found.",
        "No leading or trailing whitespace was found.",
        "The preparation operations did not change the dataset values.",
        "The manifest records file fingerprints and pipeline information; it does not verify data quality or usage rights."
    ]
}

with open(MANIFEST_PATH, "w", encoding="utf-8") as file:
    json.dump(manifest, file, indent=4)

print("Data lineage manifest created.")
print("Manifest path:", MANIFEST_PATH)

Data lineage manifest created.
Manifest path: c:\Users\M S I\VSCodeProject\AI Engineering Project\ai-engineering-talentrank-main\data\week3_manifest.json


Check the manifest

In [16]:
# Validate the data lineage manifest

with open(MANIFEST_PATH, "r", encoding="utf-8") as file:
    saved_manifest = json.load(file)

source_hash_matches = (
    saved_manifest["source"]["sha256"]
    == calculate_sha256(DATA_PATH)
)

output_hash_matches = (
    saved_manifest["output"]["sha256"]
    == calculate_sha256(PREPARED_DATA_PATH)
)

print("Source file hash matches:", source_hash_matches)
print("Output file hash matches:", output_hash_matches)
print("Source file exists:", DATA_PATH.exists())
print("Prepared file exists:", PREPARED_DATA_PATH.exists())

assert source_hash_matches
assert output_hash_matches
assert DATA_PATH.exists()
assert PREPARED_DATA_PATH.exists()

print("\nData lineage validation passed.")

Source file hash matches: True
Output file hash matches: True
Source file exists: True
Prepared file exists: True

Data lineage validation passed.
